In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [1]:

import torch
import torch.nn as nn
import numpy as np
import pandas as pd
from PIL import Image
from pathlib import Path
from torch.utils.data import Dataset, DataLoader, Subset
from torchvision import transforms

torch.manual_seed(42)
np.random.seed(42)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

data_root = Path("/kaggle/input/competitions/cifar-10-res-net-architecture-challenge-dl")
train_dir = data_root / "train_images"
train_labels = pd.read_csv(data_root / "train_labels.csv")

mean = (0.4914, 0.4822, 0.4465)
std = (0.2470, 0.2435, 0.2616)

train_transform = transforms.Compose([
    transforms.RandomCrop(32, padding=4),
    transforms.RandomHorizontalFlip(),
    transforms.RandomApply([
        transforms.ColorJitter(
            brightness=0.2,
            contrast=0.2,
            saturation=0.2,
            hue=0.04
        )
    ], p=0.8),
    transforms.ToTensor(),
    transforms.Normalize(mean, std),
    transforms.RandomErasing(
        p=0.15,
        scale=(0.02, 0.12),
        ratio=(0.3, 3.3)
    )
])

val_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(mean, std)
])

class CIFAR10CompetitionDataset(Dataset):
    def __init__(self, dataframe, image_dir, transform):
        self.dataframe = dataframe.reset_index(drop=True)
        self.image_dir = Path(image_dir)
        self.transform = transform

    def __len__(self):
        return len(self.dataframe)

    def __getitem__(self, index):
        row = self.dataframe.iloc[index]
        image_path = self.image_dir / row["filename"]

        with Image.open(image_path) as image:
            image = self.transform(image.convert("RGB"))

        label = int(row["label"])
        return image, label

train_data = CIFAR10CompetitionDataset(
    train_labels, train_dir, train_transform
)

val_data = CIFAR10CompetitionDataset(
    train_labels, train_dir, val_transform
)

indices = np.random.RandomState(42).permutation(50000)

train_set = Subset(train_data, indices[:45000])
val_set = Subset(val_data, indices[45000:])

train_loader = DataLoader(
    train_set,
    batch_size=128,
    shuffle=True,
    num_workers=2,
    pin_memory=True
)

val_loader = DataLoader(
    val_set,
    batch_size=256,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)

class BasicBlock(nn.Module):
    def __init__(self, in_channels, out_channels, stride=1):
        super().__init__()

        self.conv1 = nn.Conv2d(
            in_channels, out_channels, 3,
            stride=stride, padding=1, bias=False
        )
        self.bn1 = nn.BatchNorm2d(out_channels)

        self.conv2 = nn.Conv2d(
            out_channels, out_channels, 3,
            stride=1, padding=1, bias=False
        )
        self.bn2 = nn.BatchNorm2d(out_channels)

        if stride != 1 or in_channels != out_channels:
            self.shortcut = nn.Sequential(
                nn.Conv2d(
                    in_channels, out_channels, 1,
                    stride=stride, bias=False
                ),
                nn.BatchNorm2d(out_channels)
            )
        else:
            self.shortcut = nn.Identity()

        self.relu = nn.ReLU(inplace=True)

    def forward(self, x):
        identity = self.shortcut(x)
        out = self.relu(self.bn1(self.conv1(x)))
        out = self.bn2(self.conv2(out))
        out = self.relu(out + identity)
        return out

class DeeperResNet(nn.Module):
    def __init__(self, num_classes=10):
        super().__init__()

        self.in_channels = 64

        self.conv1 = nn.Conv2d(
            3, 64, 3, stride=1, padding=1, bias=False
        )
        self.bn1 = nn.BatchNorm2d(64)
        self.relu = nn.ReLU(inplace=True)

        self.layer1 = self.make_layer(64, 2, 1)
        self.layer2 = self.make_layer(128, 2, 2)
        self.layer3 = self.make_layer(256, 3, 2)

        self.avgpool = nn.AdaptiveAvgPool2d((1, 1))
        self.fc = nn.Linear(256, num_classes)

    def make_layer(self, out_channels, blocks, stride):
        layers = [
            BasicBlock(self.in_channels, out_channels, stride)
        ]

        self.in_channels = out_channels

        for _ in range(1, blocks):
            layers.append(
                BasicBlock(self.in_channels, out_channels)
            )

        return nn.Sequential(*layers)

    def forward(self, x):
        x = self.relu(self.bn1(self.conv1(x)))
        x = self.layer1(x)
        x = self.layer2(x)
        x = self.layer3(x)
        x = self.avgpool(x)
        x = torch.flatten(x, 1)
        x = self.fc(x)
        return x

model3 = DeeperResNet().to(device)

params3 = sum(
    p.numel() for p in model3.parameters() if p.requires_grad
)

assert device.type == "cuda", "GPU is not enabled"
assert params3 <= 5000000, "Model exceeds parameter limit"

x, y = next(iter(train_loader))

model3.eval()
with torch.no_grad():
    output = model3(x.to(device))

print("Device:", device)
print("GPU:", torch.cuda.get_device_name(0))
print("Training images:", len(train_set))
print("Validation images:", len(val_set))
print("Model parameters:", params3)
print("Under 5 million:", params3 <= 5000000)
print("Batch shape:", x.shape)
print("Output shape:", output.shape)
print("Experiment 3 model setup successful!")


Device: cuda
GPU: Tesla T4
Training images: 45000
Validation images: 5000
Model parameters: 3958346
Under 5 million: True
Batch shape: torch.Size([128, 3, 32, 32])
Output shape: torch.Size([128, 10])
Experiment 3 model setup successful!


In [2]:

import torch
import torch.nn as nn
import torch.optim as optim
import pandas as pd
from pathlib import Path

epochs3 = 50
learning_rate3 = 0.001

save_dir3 = Path("/kaggle/working/experiment3")
save_dir3.mkdir(parents=True, exist_ok=True)

best_path3 = save_dir3 / "best_deeper_resnet.pth"
last_path3 = save_dir3 / "last_checkpoint.pth"
history_path3 = save_dir3 / "training_history.csv"

criterion3 = nn.CrossEntropyLoss()

optimizer3 = optim.AdamW(
    model3.parameters(),
    lr=learning_rate3,
    weight_decay=0.0001
)

scheduler3 = optim.lr_scheduler.CosineAnnealingLR(
    optimizer3,
    T_max=epochs3
)

use_amp3 = device.type == "cuda"

scaler3 = torch.amp.GradScaler(
    "cuda",
    enabled=use_amp3
)

start_epoch3 = 0
best_acc3 = 0.0
history3 = []

if last_path3.exists():
    checkpoint3 = torch.load(
        last_path3,
        map_location=device,
        weights_only=True
    )

    model3.load_state_dict(checkpoint3["model"])
    optimizer3.load_state_dict(checkpoint3["optimizer"])
    scheduler3.load_state_dict(checkpoint3["scheduler"])
    scaler3.load_state_dict(checkpoint3["scaler"])

    start_epoch3 = checkpoint3["epoch"]
    best_acc3 = checkpoint3["best_acc"]
    history3 = checkpoint3["history"]

    print("Resuming from epoch:", start_epoch3)

print("Experiment 3 started!")
print("Device:", device)
print("Model parameters:", params3)
print("Training images:", len(train_set))
print("Validation images:", len(val_set))
print("Total epochs:", epochs3, flush=True)

for epoch in range(start_epoch3, epochs3):
    model3.train()

    train_loss = 0.0
    train_correct = 0
    train_total = 0

    for images, labels in train_loader:
        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)

        optimizer3.zero_grad(set_to_none=True)

        with torch.amp.autocast(
            "cuda",
            enabled=use_amp3
        ):
            outputs = model3(images)
            loss = criterion3(outputs, labels)

        scaler3.scale(loss).backward()
        scaler3.step(optimizer3)
        scaler3.update()

        train_loss += loss.item() * labels.size(0)
        train_correct += (
            outputs.argmax(dim=1) == labels
        ).sum().item()
        train_total += labels.size(0)

    model3.eval()

    val_loss = 0.0
    val_correct = 0
    val_total = 0

    with torch.inference_mode():
        for images, labels in val_loader:
            images = images.to(device, non_blocking=True)
            labels = labels.to(device, non_blocking=True)

            with torch.amp.autocast(
                "cuda",
                enabled=use_amp3
            ):
                outputs = model3(images)
                loss = criterion3(outputs, labels)

            val_loss += loss.item() * labels.size(0)
            val_correct += (
                outputs.argmax(dim=1) == labels
            ).sum().item()
            val_total += labels.size(0)

    avg_train_loss = train_loss / train_total
    avg_val_loss = val_loss / val_total

    train_acc = 100 * train_correct / train_total
    val_acc = 100 * val_correct / val_total

    current_lr = optimizer3.param_groups[0]["lr"]
    scheduler3.step()

    history3.append({
        "epoch": epoch + 1,
        "train_loss": avg_train_loss,
        "train_acc": train_acc,
        "val_loss": avg_val_loss,
        "val_acc": val_acc,
        "learning_rate": current_lr
    })

    if val_acc > best_acc3:
        best_acc3 = val_acc

        torch.save({
            "model": model3.state_dict(),
            "epoch": epoch + 1,
            "val_acc": val_acc,
            "parameters": params3
        }, best_path3)

        print("New best model saved!", flush=True)

    torch.save({
        "epoch": epoch + 1,
        "model": model3.state_dict(),
        "optimizer": optimizer3.state_dict(),
        "scheduler": scheduler3.state_dict(),
        "scaler": scaler3.state_dict(),
        "best_acc": best_acc3,
        "history": history3
    }, last_path3)

    pd.DataFrame(history3).to_csv(
        history_path3,
        index=False
    )

    print(
        f"Epoch [{epoch+1}/{epochs3}] | "
        f"Train Loss: {avg_train_loss:.4f} | "
        f"Train Acc: {train_acc:.2f}% | "
        f"Val Loss: {avg_val_loss:.4f} | "
        f"Val Acc: {val_acc:.2f}% | "
        f"Best: {best_acc3:.2f}%",
        flush=True
    )

print("Experiment 3 completed!")
print("Best validation accuracy:", best_acc3)
print("Best model saved to:", best_path3)


Experiment 3 started!
Device: cuda
Model parameters: 3958346
Training images: 45000
Validation images: 5000
Total epochs: 50
New best model saved!
Epoch [1/50] | Train Loss: 1.4563 | Train Acc: 46.69% | Val Loss: 1.9430 | Val Acc: 46.34% | Best: 46.34%
New best model saved!
Epoch [2/50] | Train Loss: 1.0013 | Train Acc: 64.04% | Val Loss: 1.2342 | Val Acc: 62.14% | Best: 62.14%
New best model saved!
Epoch [3/50] | Train Loss: 0.8191 | Train Acc: 70.95% | Val Loss: 0.8685 | Val Acc: 70.30% | Best: 70.30%
New best model saved!
Epoch [4/50] | Train Loss: 0.6916 | Train Acc: 75.72% | Val Loss: 0.6754 | Val Acc: 77.28% | Best: 77.28%
New best model saved!
Epoch [5/50] | Train Loss: 0.6049 | Train Acc: 78.86% | Val Loss: 0.6602 | Val Acc: 77.72% | Best: 77.72%
New best model saved!
Epoch [6/50] | Train Loss: 0.5460 | Train Acc: 81.13% | Val Loss: 0.5152 | Val Acc: 82.54% | Best: 82.54%
New best model saved!
Epoch [7/50] | Train Loss: 0.4895 | Train Acc: 83.09% | Val Loss: 0.4862 | Val Acc: 8

In [3]:

import torch
import pandas as pd
import zipfile
from pathlib import Path
from PIL import Image
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms

data_root = Path("/kaggle/input/competitions/cifar-10-res-net-architecture-challenge-dl")
test_dir = data_root / "test_images"

test_df = pd.read_csv(data_root / "test.csv")
sample_df = pd.read_csv(data_root / "sample_submission.csv")

model_path = Path("/kaggle/working/experiment3/best_deeper_resnet.pth")
output_path = Path("/kaggle/working/submission.csv")
backup_path = Path("/kaggle/working/Experiment3_backup.zip")

test_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(
        (0.4914, 0.4822, 0.4465),
        (0.2470, 0.2435, 0.2616)
    )
])

class CompetitionTestDataset(Dataset):
    def __init__(self, dataframe, image_dir, transform):
        self.filenames = dataframe["filename"].tolist()
        self.image_dir = Path(image_dir)
        self.transform = transform

    def __len__(self):
        return len(self.filenames)

    def __getitem__(self, index):
        image_path = self.image_dir / self.filenames[index]
        with Image.open(image_path) as image:
            return self.transform(image.convert("RGB"))

test_dataset = CompetitionTestDataset(
    test_df,
    test_dir,
    test_transform
)

test_loader = DataLoader(
    test_dataset,
    batch_size=128,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)

checkpoint = torch.load(
    model_path,
    map_location=device,
    weights_only=True
)

model3.load_state_dict(checkpoint["model"])
model3.eval()

predictions = []

with torch.inference_mode():
    for images in test_loader:
        images = images.to(device)
        outputs = model3(images)
        predictions.extend(outputs.argmax(dim=1).cpu().tolist())

assert len(predictions) == 2000
assert test_df["id"].tolist() == sample_df["id"].tolist()
assert all(0 <= label <= 9 for label in predictions)

submission = sample_df[["id"]].copy()
submission["label"] = predictions

assert submission.columns.tolist() == ["id", "label"]
assert not submission.isnull().any().any()

submission.to_csv(output_path, index=False)

files_to_backup = [
    Path("/kaggle/working/experiment3/best_deeper_resnet.pth"),
    Path("/kaggle/working/experiment3/last_checkpoint.pth"),
    Path("/kaggle/working/experiment3/training_history.csv"),
    output_path
]

with zipfile.ZipFile(
    backup_path,
    "w",
    zipfile.ZIP_DEFLATED
) as z:
    for file in files_to_backup:
        assert file.exists(), f"Missing file: {file}"
        z.write(file, arcname=str(file.relative_to("/kaggle/working")))

with zipfile.ZipFile(backup_path, "r") as z:
    assert z.testzip() is None

print("Best model epoch:", checkpoint["epoch"])
print("Validation accuracy:", checkpoint["val_acc"])
print("Model parameters:", checkpoint["parameters"])
print("Total predictions:", len(predictions))
print("Submission shape:", submission.shape)
print(submission.head())
print("Submission saved:", output_path)
print("Backup saved:", backup_path)
print("Backup size:", round(backup_path.stat().st_size / 1024**2, 2), "MB")
print("Experiment 3 prediction and backup completed!")


Best model epoch: 46
Validation accuracy: 93.26
Model parameters: 3958346
Total predictions: 2000
Submission shape: (2000, 2)
   id  label
0   0      6
1   1      3
2   2      5
3   3      9
4   4      3
Submission saved: /kaggle/working/submission.csv
Backup saved: /kaggle/working/Experiment3_backup.zip
Backup size: 55.59 MB
Experiment 3 prediction and backup completed!
